In [ ]:
import os
import textwrap
from pathlib import Path

from IPython.display import Markdown
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Qdrant
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from llama_parse import LlamaParse
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
llama_parse_key = os.getenv("LLAMA_PARSE")

def print_response(response):
    response_txt = response["result"]
    for chunk in response_txt.split("\n"):
        if not chunk:
            print()
            continue
        print("\n".join(textwrap.wrap(chunk, 100, break_long_words=False)))

In [8]:
# instruction = """The provided document is NVIDIA's First Quarter Fiscal 2024 Financial Results.
# This is a press release that provides detailed financial information about NVIDIA's performance for the first quarter of its fiscal year 2024.
# It includes unaudited financial statements, management's commentary, highlights of key developments, and disclosures related to NVIDIA's outlook for the next quarter.
#  The document contains many financial tables and figures. Try to be precise while answering questions based on the information in this press release."""

instruction = """The provided document contains detailed financial information, including unaudited financial statements, management commentary, key highlights, and outlook disclosures related to a company's performance for a specific fiscal period. The document may contain various financial tables, figures, and data points. When answering questions based on the information in this document, strive to provide accurate and precise
responses by carefully referencing the relevant data and details presented in the document."""

parser = LlamaParse(
    api_key=llama_parse_key,
    result_type="markdown",
    parsing_instruction=instruction,
    max_timeout=5000,
)

llama_parse_documents = await parser.aload_data("data/rpp.pdf")
# llama_parse_documents = await parser.aload_data("./data/NVIDIAAn.pdf")

Started parsing the file under job_id 4c2d63b9-b878-4762-9bb7-e241e46bee86


In [9]:
parsed_doc = llama_parse_documents[0]

In [10]:
Markdown(parsed_doc.text[:4096])

**Onion Cultivation Guide for Tamil Nadu (Beginner Friendly)**

1. **Basic Information:**
- **Crop Name:** Onion (Allium cepa)
- **Type:** Biennial (grown as annual for bulb production)
- **Growing Season:**
- Kharif: August to October
- Rabi: December to March
- **Optimal Altitude:** Grows well in plains and hilly areas up to 1200 meters above sea level.
- **Growth Duration:** Typically 4 to 5 months depending on the variety.

2. **Soil & Land Requirements:**
- **Suitable Soil Type:** Well-drained sandy loam to loamy soil with rich organic matter content.
- **Soil pH Range:** 6.0 to 7.5 (slightly acidic to neutral).
- **Soil Preparation:**
- Plough the field thoroughly to break up clods and achieve fine tilth.
- Add well-decomposed organic manure or compost during the final ploughing.
- Ensure proper drainage as onions are sensitive to waterlogging.

3. **Climate and Weather Requirements:**
- **Temperature Range:** Optimal temperature for growth is between 20°C and 25°C.
- **Rainfall Requirement:** 700 to 1000 mm during the crop cycle is ideal.
- **Sunlight:** Requires full sunlight for optimal bulb formation.
- **Wind Sensitivity:** Onions are not very sensitive to wind but avoid high-speed winds during flowering.

4. **Water Requirements & Irrigation:**
- **Water Needs:** Onion is moderately water-intensive and requires consistent moisture, especially during bulb formation.
- **Recommended Irrigation Methods:**
- **Drip Irrigation:** Most efficient method for onions, saves water and improves yield.
- **Furrow Irrigation:** Traditional method, but requires good water management to avoid over-watering.
- **Irrigation Schedule:** [Details not provided in the text]

This guide provides essential information for beginners looking to cultivate onions in Tamil Nadu, covering key aspects such as soil, climate, and irrigation needs.

In [11]:
document_path = Path("data/parsed_document.md")
with document_path.open("a") as f:
    f.write(parsed_doc.text)

In [13]:
loader = UnstructuredMarkdownLoader(document_path)
loaded_documents = loader.load()

In [14]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=128)
docs = text_splitter.split_documents(loaded_documents)
len(docs)

61

In [15]:
#Downloading embeddings
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-base-en-v1.5")

c:\Users\vishn\Desktop\Programs\CodeOClock\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:30<00:00,  6.04s/it]


In [16]:
qdrant = Qdrant.from_documents(
    docs,
    embeddings,
    # location=":memory:",
    path="./db",
    collection_name="document_embeddings",
)

In [17]:
%%time
# query = "What is the most important innovation from Nvidia?"
query = "What is the most important innovation from Nvidia?"
similar_docs = qdrant.similarity_search_with_score(query)

CPU times: total: 766 ms
Wall time: 205 ms


In [18]:
for doc, score in similar_docs:
    print(f"text: {doc.page_content[:256]}\n")
    print(f"score: {score}")
    print("-" * 80)
    print()

text: cm apart. Irrigation: Ensure adequate moisture in the soil. Week 2-4: Seedling Stage: Weed Management: Hand-weed or apply mulch to suppress weeds. Irrigation: Maintain consistent soil moisture. Week 5-8: Vegetative Growth: Fertilizer Application: Apply the

score: 0.40341463144604484
--------------------------------------------------------------------------------

text: cm apart. Irrigation: Ensure adequate moisture in the soil. Week 2-4: Seedling Stage: Weed Management: Hand-weed or apply mulch to suppress weeds. Irrigation: Maintain consistent soil moisture. Week 5-8: Vegetative Growth: Fertilizer Application: Apply the

score: 0.40341463144604484
--------------------------------------------------------------------------------

text: cm apart. Irrigation: Ensure adequate moisture in the soil. Week 2-4: Seedling Stage: Weed Management: Hand-weed or apply mulch to suppress weeds. Irrigation: Maintain consistent soil moisture. Week 5-8: Vegetative Growth: Fertilizer Application: A

In [19]:
%%time
retriever = qdrant.as_retriever(search_kwargs={"k": 5})
retrieved_docs = retriever.invoke(query)

CPU times: total: 625 ms
Wall time: 179 ms


In [20]:
for doc in retrieved_docs:
    print(f"id: {doc.metadata['_id']}\n")
    print(f"text: {doc.page_content[:256]}\n")
    print("-" * 80)
    print()

id: b0fe72ec3e9349429db5ff0852f3ff64

text: cm apart. Irrigation: Ensure adequate moisture in the soil. Week 2-4: Seedling Stage: Weed Management: Hand-weed or apply mulch to suppress weeds. Irrigation: Maintain consistent soil moisture. Week 5-8: Vegetative Growth: Fertilizer Application: Apply the

--------------------------------------------------------------------------------

id: b49ab8a7c2f348f5a9b531c0cb50d2ae

text: cm apart. Irrigation: Ensure adequate moisture in the soil. Week 2-4: Seedling Stage: Weed Management: Hand-weed or apply mulch to suppress weeds. Irrigation: Maintain consistent soil moisture. Week 5-8: Vegetative Growth: Fertilizer Application: Apply the

--------------------------------------------------------------------------------

id: 94b74c91b5994162aa26b14f68efc042

text: cm apart. Irrigation: Ensure adequate moisture in the soil. Week 2-4: Seedling Stage: Weed Management: Hand-weed or apply mulch to suppress weeds. Irrigation: Maintain consistent soil mois

In [2]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors.rankllm_rerank import RankLLMRerank

compressor = RankLLMRerank(top_n=3, model="zephyr")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

ImportError: Could not import rank_llm python package. Please install it with `pip install rank_llm`.

In [22]:
%%time
reranked_docs = compression_retriever.invoke(query)
len(reranked_docs)

NameError: name 'compression_retriever' is not defined

In [ ]:
for doc in reranked_docs:
    print(f"id: {doc.metadata['_id']}\n")
    print(f"text: {doc.page_content[:256]}\n")
    print(f"score: {doc.metadata['relevance_score']}")
    print("-" * 80)
    print()

In [23]:
llm = ChatGroq(temperature=0, model_name="llama3-70b-8192")

In [24]:
prompt_template = """
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Answer the question and provide additional helpful information,
based on the pieces of information, if applicable. Be succinct.

Responses should be properly formatted to be easily read.
"""

prompt = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

In [25]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=compression_retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt, "verbose": True},
)

NameError: name 'compression_retriever' is not defined

In [ ]:
%%time
response = qa.invoke("What is the most significant innovation from Nvidia?")

In [ ]:
print_response(response)

In [ ]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=compression_retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt, "verbose": False},
)

In [ ]:
%%time
response = qa.invoke("What is the revenue for 2024 and % change?")

In [ ]:
Markdown(response["result"])

In [ ]:
%%time
response = qa.invoke("What is the FY23?")

In [ ]:
print_response(response)

In [ ]:
%%time
response = qa.invoke(
    "How much is the revenue minus the costs and expenses for 2024? Calculate the answer"
)

In [ ]:
print_response(response)

In [ ]:
%%time
response = qa.invoke("compare the Gross profit from 2022 and 2023?")

In [ ]:
Markdown(response["result"])

In [ ]:
%%time
response = qa.invoke("what is the share holding of board of directors Mr. P Arulsundaram")

In [ ]:
Markdown(response["result"])